<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Object Detection

Machine-learned object detection models can be extremely useful. They are faster and often more reliable than traditional computer vision models. Additionally, we can use pretrained model weights to cut down immensely on training time.

Here's an example of what an object detector might output:



<iframe width="800" height="500"
src="https://www.youtube.com/embed/3jD02dxL6gg" 
frameborder="0" 
allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" 
allowfullscreen
style="margin: auto; display: block"></iframe>


In this exercise, you will create your own Duckietown object detection dataset. You will learn about the general structure such a dataset should follow. You will train the object detection model on that dataset [in a subsequent notebook](../03-Training/training.ipynb), and finally export the model from PyTorch to ONNX for easier deployment across platforms. Finally, you will integrate the model and test the integration [in the last notebook](../04-Integration/integration.ipynb), so that your Duckiebot knows how to recognize duckie pedestrians (and thus avoid them). You can test your object detector in simulation and on your real Duckiebot.




## Setup

First, we need some global variables. These allow you to change the directory where we store the data you will need. You can also change the image size to reflect what your final model uses, but you can worry about that later.

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
DATA_DIR="../../assets/data/"
# this is the percentage of data that will go into the training set (as opposed to the testing set)
TRAIN_TEST_SPLIT_PERCENTAGE = 0.8

We will describe the overall process that you follow [next](#model-training-workflow). Then, you will build your own dataset with simulated and real images in the [data collection section](#data-collection).

## Model Training Workflow

What does an object detection dataset look like? Clearly, the specifics will depend on the convention used by specific models, but the general idea is intuitive:

- We need an **image**
- Each image may contain **multiple objects**
- For each object, we need:
  - a **bounding box** (where it is)
  - a **class label** (what it is)

### Bounding box formats

There are several common ways to represent bounding boxes:

- `x_min y_min width height`
- `x_min y_min x_max y_max`
- `x_center y_center width height`

In this exercise, we will use a YOLO-style model (YOLOv11 from the [Ultralytics](https://github.com/ultralytics/ultralytics) library), which expects bounding boxes in the format:
> `x_center y_center width height`

All four values are **normalized to \[0, 1\]** with respect to the image width and height. Each label file (`.txt`) contains one line per object.

How are the bounding boxes defined?

![image of a bounding box](../../assets/images/bbox.png)



### How we obtain bounding boxes

In a traditional object detection pipeline, you would need to label a dataset of images **by hand**.  However, since this is quite laborious and the images that we are going to collect should be relatively simple, we will try to generate labels automatically using [**SAM3**](https://github.com/facebookresearch/sam3), a foundation model capable of producing high-quality annotations for objects in an image or videos. We will only be using it to label the images we will collect during the data-collection step.

You can take a look at Pytorch's object detection [tutorial](https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html) to read more on how to obtain bounding boxes.


The auto-labelling process works as follows:

1. **Collect RGB images**  
   Images come from the robot (virtual or real).

2. **Run SAM3 to obtain instance bounding box**  
   SAM3 predicts a set of bounding boxes in the image based on the prompts or pre-defined classes. Each bounding box corresponds to a visually distinct object.
   This produces `(x_min, y_min, x_max, y_max)` in pixel coordinates which we then process to match the format expected by **Ultralytics** for training. 

3. **Duckietown classes of interest**  
   The goal of this exercise is to make Duckietown safer: we want to be able to detect duckie pedestrians on the road and avoid squishing them. 
   Common classes you may care about are:  
   - duckie  
   - cone  
   - truck  
   - bus 
   - duckiebot

   You will need to set up a prompt for classes you might be interested in. For the rest of the notebooks we only care about detecting duckies but you are welcome to create your detector with as many objects as you may need.

5. **Convert boxes to YOLO format**  
   We have provided a function to post-process the collected labels to create a dataset for you to train your duckie detector once the auto-labelling step is complete.

In the next sections, we will explain the steps to generate data.

## Data collection


### Tool and Format


In `data_collection` mode, the agent saves streaming camera data while you drive the robot.


Your dataset's format depends heavily on your model. In this case you are going to use an [Ultralytics](https://github.com/ultralytics/ultralytics), so you should closely follow their [guide on how to train using custom data](https://docs.ultralytics.com/modes/train/). Your data should follow the following directory structure:

![image of dataset save format](../../assets/images/dataset_format.png)

The dataset will be called `duckietown_dataset` and will be stored in the `/data` directory on your robot, but will be later copied inside the [assets/data](../../assets/data/) directory of this learning experience. 

We will create two subdirectories in that folder: `train` and `val`. In each of these directories we will create two subdirectories, `images` and `labels`. Inside `images`, you must place your images, and inside `labels`, you must place the images' bounding boxes data. Notice that the label files use the same name as their corresponding image files but with a different extension. In other words, the data for `0.jpg` can be found in `0.txt`.

The format for the label files is fairly simple. For each bounding box in the corresponding image, write a row of the form `class x_center y_center width height`. Keep in mind that the pixel data must be 0-to-1 normalized (i.e., you can calculate the usual `x_center y_center width height` in pixel space and divide by the image's size). For example,

    0 0.5 0.5 0.2 0.2
    1 0.60 0.70 0.4 0.2

this says "there is a duckie (class 0) centered in the image, whose width and height are 20% of the image's. There is also a cone (class 1) whose center is at 60% of the image's maximal x value and 70% of the image's maximal y value, and its width is 40% of the image's own while its height is 20%."

Again, it is recommended that you read the guide posted on Ultralytics: [guide on how to train using custom data](https://docs.ultralytics.com/modes/train/).

### Running data collection

We now use the `data_collection` launcher provided to collect data in both simulation and real environments. This launcher runs the agent in data collection mode which automatically saves camera images to the dataset directory as you drive around.

*If you are running on a virtual robot*: make sure you have created a virtual Duckiebot and started the Duckiematrix using the [procedure described in the README](../../README.md).


* First, let's build the code:

    ```shell
    dts code build -R ROBOT_NAME
    ```


* Then we can run the code and specify the `data-collection` launcher:

    ```shell
    dts code workbench [-m] -R ROBOT_NAME -L data_collection
    ```

    This starts the data-collection agent where you should include the `-m` flag if you are running the Duckiematrix.

    You can open the joystick with

    ```shell
    dts duckiebot keyboard_control ROBOT_NAME
    ```

    and pilot the robot around while getting lots of good views of duckies (In the Duckiematrix you can also achieve this with the `w`, `a`, `s` and `d` keys directly in the Duckiematrix window as described in the [README](../../README.md) if you prefer).


The agent will:

- capture up to 1000 RGB images,

- save them under `data/data_collection/` directory,

- then stop collecting images automatically.

You can also stop it manually at any time with `CTRL-C`.


### Copy the data logs to your local machine

The logs that you collected are stored on your robot in the `/data/data_collection/` directory. You can view the images you collected through the Dashboard  by pointing your browser to ROBOT_NAME.local (can be real or virtual) and then clicking on `File Manager` on the left hand side. From there, you can navigate to `/data/data_collection/`. 

Ultimately, we will want the data to be stored on your local computer so that we can easily upload it to Google Drive in the next step. You can download it through the Dashboard or you can do it through the command line (this command should run from the root directory of this repo outside of the VSCode environment):

```shell
scp -r duckie@ROBOT_NAME.local:/data/data_collection assets/data/data_collection
```

If you are asked for a password it is the usual `quackquack`. 

In the file Explorer on the left (in this VSCode environment), you should now see a folder `data_collection`  inside the folder `assets/data` and inside you should see all of the `.png` files for the images you collected.

Once you've confirmed that the data was copied successfully, proceed to the next step.

### Combining with the datasets & training/test set splits

When training supervised learning models, one must worry about overfitting to the training set. If you can keep *some* of your dataset *out* of your training data, you can use it to verify that your model does not overfit to your dataset by *testing* it on the data you left out. We call this chunk of data the *validation set*. 

You can experiment with the `TRAIN_TEST_SPLIT_PERCENTAGE` variable defined at the top of this notebook. Tune its value to adjust the percentage of the data that is used for training as opposed to testing.

Once the data-collection launcher has finished, you will have a single `data_collection` containing all captured RGB images. Before these images are auto-labelled using SAM3, we need to split them into **training** and **validation** subsets in a format that YOLO expects.

In [5]:
import os
import random
from pathlib import Path
import shutil

def create_train_val_split(data_collection_dir, dataset_dir, split_percentage):
    data_collection_dir = Path(data_collection_dir)
    dataset_dir = Path(dataset_dir)

    train_img_dir = dataset_dir / "train" / "images"
    train_lbl_dir = dataset_dir / "train" / "labels"
    val_img_dir   = dataset_dir / "val" / "images"
    val_lbl_dir   = dataset_dir / "val" / "labels"

    train_img_dir.mkdir(parents=True, exist_ok=True)
    train_lbl_dir.mkdir(parents=True, exist_ok=True)
    val_img_dir.mkdir(parents=True, exist_ok=True)
    val_lbl_dir.mkdir(parents=True, exist_ok=True)

    images = sorted(
        p for p in data_collection_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )

    random.shuffle(images)
    split_idx = int(len(images) * split_percentage)

    train_images = images[:split_idx]
    val_images   = images[split_idx:]

    def copy_and_init_label(img_path, dest_img_dir, dest_lbl_dir=None):
        shutil.copy(img_path, dest_img_dir / img_path.name)
        # (dest_lbl_dir / (img_path.stem + ".txt")).touch()  # we can either create empty label files or ignore it

    for img in train_images:
        copy_and_init_label(img, train_img_dir, train_lbl_dir)

    for img in val_images:
        copy_and_init_label(img, val_img_dir, val_lbl_dir)

    print(f"Created train/val split: {len(train_images)} train, {len(val_images)} val")


In [6]:
create_train_val_split(
    data_collection_dir=f"{DATA_DIR}/data_collection",
    dataset_dir=f"{DATA_DIR}/duckietown_dataset",
    split_percentage=TRAIN_TEST_SPLIT_PERCENTAGE,
)

Created train/val split: 800 train, 200 val


At this point you should see an additional folder in your [assets/data](../../assets/data) directory called [duckietown_dataset](../../assets/data/duckietown_dataset/) which has the `train` and `val` split that we described above.

**Note**: You probably do not want to push contents of the `assets/data` directory to Git since it contains a lot of large files and this shouldn't be necessary. 

# Next step

You are now ready to move on to dataset preparation and training. You can continue with the [Training notebook](../03-Training/training.ipynb)